# "Covid-19 - India Dashboard"
> "Some charts exploring Covid-19 India Data"

- toc: false 
- badges: false
- comments: true
- categories: [ai-in-society]
- image: images/coronavirus.jpg

# Covid-19 India Timeline

In [ ]:
#hide
from IPython.display import HTML, Javascript, display
from string import Template
import numpy as np
import pandas as pd
import json
import math
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
import time
import requests

In [ ]:
#hide
mpl.rcParams['figure.dpi'] = 100
display(Javascript("require.config({paths: {d3: 'https://d3js.org/d3.v5.min.js'}});"));

In [ ]:
#hide
official_history_url = "https://api.rootnet.in/covid19-in/stats/history" 
data_raw = requests.get(official_history_url).text

In [ ]:
#hide
data = json.loads(data_raw)['data']

In [ ]:
#hide
js_data = f'let dataset = {data}'

In [ ]:
#hide
html_temp = Template('''
    <script src = "https://d3js.org/d3.v5.min.js"></script> 
    <style scoped>
        $css_text
    </style>
    <div id="wrapper" class="wrapper">

            <div id="tooltip" class="tooltip">
                <div class="tooltip-date">
                    <span id="date"></span>
                </div>
                <div class="tooltip-confirmed">
                    Confirmed Cases: <span id="confirmed"></span>
                </div>
            </div>
        </div>
    <script>
    $dataset
    $d3_script
    </script>
''')

In [ ]:
#hide
css_text = '''
.wrapper {
    position: relative;
}

.line {
    fill: none;
    stroke: #FF9933;
    stroke-width: 2;
}

.y-axis-label {
    fill: black;
    font-size: 1.4em;
    text-anchor: middle;
    transform: rotate(-90deg);
}

.listening-rect {
    fill: transparent;
}

body {
    display: flex;
    justify-content: center;
    padding: 5em 1em;
    font-family: sans-serif;
}

.tooltip {
    opacity: 0;
    position: absolute;
    top: -14px;
    left: 0;
    padding: 0.6em 1em;
    background: #fff;
    text-align: center;
    line-height: 1.4em;
    font-size: 0.9em;
    border: 1px solid #ddd;
    z-index: 10;
    transition: all 0.1s ease-out;
    pointer-events: none;
}

.tooltip:before {
    content: '';
    position: absolute;
    bottom: 0;
    left: 50%;
    width: 12px;
    height: 12px;
    background: white;
    border: 1px solid #ddd;
    border-top-color: transparent;
    border-left-color: transparent;
    transform: translate(-50%, 50%) rotate(45deg);
    transform-origin: center center;
    z-index: 10;
}

.tooltip-date {
    margin-bottom: 0.2em;
    font-weight: 600;
    font-size: 1.1em;
    line-height: 1.4em;
}
'''

In [ ]:
#hide
d3_script = '''
function drawLineChart() {
  
  yAccessor = d => d.summary.total
  const dateParser = d3.timeParse("%Y-%m-%d")
  format = d3.timeFormat("%b-%d")
  const xAccessor = d => dateParser(d.day)
  dataset = dataset.sort((a,b) => xAccessor(a) - xAccessor(b)).slice(0, 100)

  let dimensions = {
    width: window.innerWidth * 0.8,
    height: 400,
    margin: {
      top: 15,
      right: 15,
      bottom: 40,
      left: 60,
    },
  }
  dimensions.boundedWidth = dimensions.width - dimensions.margin.left - dimensions.margin.right
  dimensions.boundedHeight = dimensions.height - dimensions.margin.top - dimensions.margin.bottom

  // 3. Draw canvas

  const wrapper = d3.select("#wrapper")
    .append("svg")
      .attr("width", dimensions.width)
      .attr("height", dimensions.height)
      
    wrapper.append("rect")
        .attr("class", "chart-background")
        .attr("width", "100%")
        .attr("height", "100%")
        .attr("fill", "#faebd7")

  const bounds = wrapper.append("g")
      .attr("transform", `translate(${
        dimensions.margin.left
      }, ${
        dimensions.margin.top
      })`)
      
    bounds.append("text")
      .attr("class", "chart-watermark")
      .attr("x", dimensions.width - 200)
      .attr("y", dimensions.height - 60)
      .attr("fill", "grey")
      .style("opacity", 0.5)
      .html("@Alepthoughts")

  bounds.append("defs").append("clipPath")
      .attr("id", "bounds-clip-path")
    .append("rect")
      .attr("width", dimensions.boundedWidth)
      .attr("height", dimensions.boundedHeight)

  const clip = bounds.append("g")
    .attr("clip-path", "url(#bounds-clip-path)")

  // 4. Create scales

  const yScale = d3.scaleLinear()
    .domain(d3.extent(dataset, yAccessor))
    .range([dimensions.boundedHeight, 0])
    .nice()

  
  const xScale = d3.scaleTime()
    .domain(d3.extent(dataset, xAccessor))
    .range([0, dimensions.boundedWidth])

  const lineGenerator = d3.line()
    .x(d => xScale(xAccessor(d)))
    .y(d => yScale(yAccessor(d)))

  const line = clip.append("path")
      .attr("class", "line")
      .attr("d", lineGenerator(dataset))

  const yAxisGenerator = d3.axisLeft()
      .scale(yScale)
      
  
  const yAxis = bounds.append("g")
      .attr("class", "y-axis")
      .call(yAxisGenerator)

  const yAxisLabel = yAxis.append("text")
      .attr("class", "y-axis-label")
      .attr("x", -dimensions.boundedHeight / 2)
      .attr("y", -dimensions.margin.left + 10)
      .html("Confirmed Cases")

      const xAxisGenerator = d3.axisBottom()
      .scale(xScale)
      .tickFormat(format)
  
  const xAxis = bounds.append("g")
      .attr("class", "x-axis")
      .style("transform", `translateY(${dimensions.boundedHeight}px)`)
      .call(xAxisGenerator)

  const listeningRect = bounds.append("rect")
      .attr("class", "listening-rect")
      .attr("width", dimensions.boundedWidth)
      .attr("height", dimensions.boundedHeight)
      .on("mousemove", onMouseMove)
      .on("mouseleave", onMouseLeave)

  const tooltip = d3.select("#tooltip")
  const tooltipCircle = bounds.append("circle")
          .attr("class", "tooltip-circle")
          .attr("r", 4)
          .attr("stroke", "#FF9933")
          .attr("fill", "white")
          .attr("stroke-width", 2)
          .style("opacity", 0)

  function onMouseMove() {
    const mousePosition = d3.mouse(this)
    const hoveredDate = xScale.invert(mousePosition[0])
        
    const getDistanceFromHoveredDate = d => Math.abs(xAccessor(d) - hoveredDate)
    const closestIndex = d3.scan(dataset, (a, b) => (
    getDistanceFromHoveredDate(a) - getDistanceFromHoveredDate(b)
    ))
    const closestDataPoint = dataset[closestIndex]
        
    const closestXValue = xAccessor(closestDataPoint)
    const closestYValue = yAccessor(closestDataPoint)
        
    const formatDate = d3.timeFormat("%B %A %-d, %Y")
    tooltip.select("#date")
    .text(formatDate(closestXValue))

    tooltip.select("#confirmed")
        .html(closestYValue)

    const x = xScale(closestXValue)
      + dimensions.margin.left
    const y = yScale(closestYValue)
      + dimensions.margin.top

    tooltip.style("transform", `translate(`
      + `calc( -50% + ${x}px),`
      + `calc(-100% + ${y}px)`
      + `)`)

    tooltip.style("opacity", 1)

    tooltipCircle
        .attr("cx", xScale(closestXValue))
        .attr("cy", yScale(closestYValue))
        .style("opacity", 1)
  }
  function onMouseLeave() {
    tooltip.style("opacity", 0)

    tooltipCircle.style("opacity", 0)
  }
      
}
drawLineChart();
'''

In [ ]:
#hide
html_text = html_temp.substitute({
    'css_text': css_text,
    'd3_script' : d3_script,
    'dataset':js_data
})

In [ ]:
#hide_input
HTML(html_text)